<a href="https://colab.research.google.com/github/threaddotsleep/senilebot/blob/main/senile_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -q torch transformers accelerate bitsandbytes gradio

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configure 4-bit quantization with CPU offloading enabled inside the config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True
)

# Load model with the updated configuration
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
import gradio as gr

def predict(message, history):
    """Predict function for the ChatInterface.

    Handles messages/history formatting safely across different Gradio versions.
    """
    system_prompt = (
        "You are an elderly character experiencing mild-to-moderate cognitive decline. "
        "Mix up timelines, confuse modern things with mid-20th-century life, lose your train "
        "of thought mid-sentence, and occasionally pivot to unrelated memories or ask repetitive questions. "
        "Speak lots of non-sensical gibberish when responding to the point of unsettling. "
        "Never break character."
    )

    messages = [{"role": "system", "content": system_prompt}]

    def clean_content(content):
        """Helper to safely extract string text from diverse Gradio structures."""
        if isinstance(content, str):
            return content
        if isinstance(content, dict):
            if "text" in content:
                return str(content["text"])
            elif "content" in content:
                return clean_content(content["content"])
            return ""
        if isinstance(content, list):
            # Recursively find and join any text items
            extracted_texts = []
            for item in content:
                if isinstance(item, dict) and item.get("type") == "text":
                    extracted_texts.append(str(item.get("text", "")))
                elif isinstance(item, str):
                    extracted_texts.append(item)
                else:
                    cleaned = clean_content(item)
                    if cleaned:
                        extracted_texts.append(cleaned)
            return " ".join(extracted_texts)
        return str(content) if content is not None else ""

    # Map history to standard chat model roles
    for interaction in history:
        if isinstance(interaction, dict):
            role = interaction.get("role")
            content = clean_content(interaction.get("content"))
            if role and content:
                messages.append({"role": role, "content": content})
        elif isinstance(interaction, (list, tuple)) and len(interaction) == 2:
            user_content, assistant_content = interaction
            messages.append({"role": "user", "content": clean_content(user_content)})
            messages.append({"role": "assistant", "content": clean_content(assistant_content)})

    # Append current message
    messages.append({"role": "user", "content": clean_content(message)})

    # Generate completion
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = pipe(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.85,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )

    response = outputs[0]['generated_text'][len(prompt):].strip()
    return response

In [ ]:
with gr.Blocks() as demo:
    gr.ChatInterface(
        fn=predict,
        title="SenileBot",
        description="A chat persona exploring memory, time, and meandering stories."
    )

# share=True creates a public gradio.live link running directly from your Colab GPU session
demo.launch(share=True, debug=True, theme="soft")

In [ ]:
import gradio as gr
gr.close_all()